## Integrated Decision Memo

Felipe Tasama · 11/20/2026 · Dr. Zachary Smith · FIN-450 · Case 4

### Coca-Cola Option Pricing and Risk Analysis

#### Equity Valuation

I used the November 19, 2026, closing price of $96.09 for Coca-Cola. It was above both my September 24 DCF estimate of $84.36 per diluted share and my comparable company estimate of $83.35. The DCF depends on the assumptions about future cash flows, the discount rate, and long–term growth. It was also completed approximately two months prior to the stock price that was used in my project. I applied the trailing P/E from PepsiCo and Keurig Dr Pepper to Coca-Cola's $3.40 diluted EPS.

The 11 analyst targets gave me a more promising image. The average was $100.27 and the median was $100.00. The targets had ranged from $89 to $112, with a standard deviation of $6.13, but half were between $97 and $103. One setback is that the targets look about two months beyond the option’s expiration. This is important because it could reach an analyst's target after the call has already expired.

#### Option Valuation

I analyzed an American $95 call expiring September 17, 2027, with a multiplier of 100 shares. The five possible stock prices at expiration were $92, $97, $100, $103, and $109. With weights of 10%, 20%, 40%, 20%, and 10%, the average payoff was $5.40 per share before discounting and $5.1975 after discounting. Four of the five scenarios produced a payoff, but several outcomes were close to the strike, which kept the expected payoff low.

The 200-step Cox–Ross–Rubinstein (CRR) model valued the call at $9.0064 per share, within the $8.60 bid and $10.55 ask and below the $9.575 midpoint. My analyst scenarios use subjective weights on a narrow set of terminal prices, while the CRR model uses risk-neutral probabilities and 22.14% implied volatility. The tree assumes a 2.21% continuous dividend yield and allows early exercise, which added about $0.000002 per share to the European benchmark in this run.

#### Greeks and Risk Profile

The stock price is the first thing I would watch because KO was only $1.09 above the strike. A delta of 0.590292 means that raising KO by $1 would add about $59. Gamma of 0.019620 means that same move would raise delta by about 0.02. According to my chart, gamma was highest near $87.54 at the current maturity. Closer to its expiration, we can expect delta to change more sharply around the $95 strike. Vega was $0.331425 per share of a one percentage point increase in implied volatility, or about $33 per contract. Theta was -$0.014776 per share per calendar day, so one day passing would cost about $1.48 per contract. Rho was $0.394738 per share for a one percentage point increase in rates. For a 10-basis-point increase, the model estimated a gain of about $3.95 per contract.

#### Approximation and Model Risk

When I raised KO by $1, delta alone estimated that it would have a gain of about $0.5903 per share. Adding gamma brought the estimate to $0.6001, slightly closer to the $0.5958 gain from rerunning the model. When I also raised volatility by one percentage point, let one calendar day pass, and raised rates by 10 basis points, the estimated gain was $0.9562 versus $0.9525 from full repricing. I would put less weight on the approximation for larger moves because the Greeks change as market conditions change.

The Gamma estimate changed depending on how much the stock price was moved in the model. Using a $1.92 move, I got a gamma of 0.0160. Using a $2.88 move, I got 0.0194. Since both estimates differ from each other, I am treating Gamma as an approximation. I also updated the deployment workflow so that it exports the model results after the notebook runs. That way the dashboard results can show the results from the Coca-Cola analysis.

#### Conclusion

In conclusion, the model put it at $9.0064 per share, within the $8.60 to $10.55 quoted range. With only 12 contracts traded that day, and a $1.95 gap between the bid and ask, it’s hard to tell where the next trade would land. I would change my view if the model was updated with a fresh set of market inputs, and its estimate came in below that day's bid or above the asking price.


# Valuation-to-Option Pricing Lab

Two tracks remain distinct: a belief-based analyst-scenario payoff and a CRR no-arbitrage value.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')


In [ ]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}; weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({'state':list(quantile_map),'quantile':list(quantile_map.values()),'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],'subjective_weight':[weights[k] for k in quantile_map]})
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float); return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type']); states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum()); discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years'])); belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum()); weight_zero=1-weight_positive


In [ ]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]; conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_disp=float(np.log(targets.target/M['spot']).std(ddof=1)); crr_disp=float(M['volatility']*np.sqrt(M['maturity_years']))
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
results={'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},'distribution_comparison':{'analyst_log_dispersion':analyst_disp,'crr_horizon_volatility':crr_disp,'message':'Analyst targets are narrower than the volatility input used by CRR.' if analyst_disp<crr_disp else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'},'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']},'analyst_horizon_years':float(targets.horizon_months.iloc[0]/12)}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2)); results['comparison']


## Model revision history

The original five-state scenario and CRR notebook is preserved in Git history, including the starting validation checks. The deployment workflow was improved to export the full model after executing the notebook, so the dashboard receives the documented KO results. The Greeks extension adds same-lattice sensitivities, finite-difference comparisons, full repricing, and Plotly sensitivity charts.

In [ ]:
from IPython.display import display
from scripts.model import build_results

analysis = build_results()
display(pd.DataFrame([analysis['valuation'] | {'Analyst mean': analysis['analyst_stats']['mean']}]))
display(pd.DataFrame([analysis['analyst_stats']]))
display(targets)
display(pd.DataFrame(analysis['states']))
display(pd.DataFrame([analysis['belief_decomposition']]))
display(pd.DataFrame([{
    'American CRR': analysis['lattice']['price'],
    'European benchmark': analysis['european_price'],
    'Early exercise premium': analysis['exercise_premium'],
    'American bounds pass': analysis['american_bounds_pass'],
    'Delta range pass': analysis['delta_check'],
    'Gamma sign pass': analysis['gamma_check'],
    'U': analysis['lattice']['u'],
    'D': analysis['lattice']['d'],
    'Risk-neutral probability': analysis['lattice']['p_star'],
    'Analyst horizon years': analysis['analyst_horizon_years'],
    'Option maturity years': M['maturity_years'],
}]))
display(pd.DataFrame(analysis['convergence']))
display(pd.DataFrame(analysis['greek_validation']))
display(pd.DataFrame(analysis['single_factor']))
display(pd.DataFrame(analysis['repricing']))

## AI-use disclosure

ChatGPT assisted with sourcing, code changes, numerical checks, and drafting. The dated sources are recorded in the CSVs, and the notebook includes finite-difference comparisons and full repricing for verification. Review the sources, assumptions, and wording before submitting and update this disclosure to reflect your own verification and revisions.
